# FPL News Feature Pipeline

This notebook extracts NLP features from football news articles to improve FPL predictions.

## Research Background

This approach is based on two key papers:

1. **ESPN Fantasy Football with IBM Watson** (2021)
   - Processed 2.3M articles/day
   - Used entity linking + embeddings
   - Significant prediction improvement

2. **FPL Transformer Paper** (2025)
   - Used sentiment analysis on news
   - Found negative sentiment correlates with lower points
   - Achieved 10% MSE reduction

## Pipeline Overview

```
Guardian API  →  Clean Text  →  Link to Players  →  Extract Features  →  Aggregate
   (scrape)       (parse)        (entity link)      (sentiment +        (per player
                                                     embeddings)          per GW)
```

## Output

Final output: `news_features.csv` with columns:
- `element` - FPL player ID
- `season` - Season string
- `news_article_count` - Number of articles mentioning player
- `news_sentiment_positive/negative/neutral` - Average sentiment
- `news_emb_0` to `news_emb_383` - 384-dimensional semantic embedding

---

## 1️⃣ Setup & Installation

This cell:
1. Installs required packages (only on Colab)
2. Downloads the spaCy language model for entity recognition
3. Sets up file paths
4. Prompts you to upload your FPL data

In [ ]:
# ============================================================
# SETUP - Run this cell first
# ============================================================

import sys
import os

# Detect if running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab - Installing packages...")
    
    # Install required packages
    # - spacy: Named Entity Recognition (finding player names)
    # - sentence-transformers: Text embeddings (semantic meaning)
    # - transformers: Sentiment analysis models
    # - beautifulsoup4: HTML parsing
    !pip install -q spacy sentence-transformers transformers beautifulsoup4 lxml
    
    # Download spaCy English model (small version - fast but accurate enough)
    !python -m spacy download en_core_web_sm -q
    
    # Create directory structure
    BASE_PATH = '/content/FYP'
    os.makedirs(f'{BASE_PATH}/data/processed/merged', exist_ok=True)
    os.makedirs(f'{BASE_PATH}/data/raw/news', exist_ok=True)
    os.makedirs(f'{BASE_PATH}/data/features', exist_ok=True)
    
    # Upload FPL data
    print("\n📁 Please upload your fpl_base.csv file:")
    from google.colab import files
    uploaded = files.upload()
    
    # Move to correct location
    for filename in uploaded.keys():
        if 'fpl' in filename.lower():
            os.rename(filename, f'{BASE_PATH}/data/processed/merged/fpl_base.csv')
            print(f"✓ Moved {filename} to data/processed/merged/")
else:
    print("💻 Running locally")
    BASE_PATH = '..'  # Assumes notebook is in notebooks/ folder

print(f"\n✓ Setup complete! Base path: {BASE_PATH}")

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================

import json
import re
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

# NLP imports
import spacy
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DATA_PATH = Path(BASE_PATH) / "data"
RAW_NEWS_DIR = DATA_PATH / "raw" / "news"
FPL_DATA_PATH = DATA_PATH / "processed" / "merged" / "fpl_base.csv"
OUTPUT_PATH = DATA_PATH / "features" / "news_features.csv"

# Create directories
RAW_NEWS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Guardian API (free tier)
# Get your own key at: https://bonobo.capi.gutools.co.uk/register/developer
GUARDIAN_API_KEY = os.environ.get("GUARDIAN_API_KEY", "test")  # 'test' key has limits

# Seasons to process (based on research: 2 seasons is optimal)
SEASONS = {
    "2023-24": ("2023-08-01", "2024-05-31"),  # Complete season
    "2024-25": ("2024-08-01", "2025-05-31"),  # Current season
}

# Model settings
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # Fast, 384 dimensions
EMBEDDING_DIM = 384

# Entity linking settings
MIN_NAME_LENGTH = 4  # Skip very short names like "Son" to avoid false positives

print("✓ Configuration loaded!")
print(f"  Seasons: {list(SEASONS.keys())}")
print(f"  FPL data exists: {FPL_DATA_PATH.exists()}")
print(f"  Guardian API key: {'custom' if GUARDIAN_API_KEY != 'test' else 'test (limited)'}")

---

## 2️⃣ Scrape Articles from Guardian API

**What is the Guardian API?**

The Guardian newspaper provides a free API for accessing their articles. Key features:
- Historical access (back to 1999)
- Can filter by tags (e.g., `football/premierleague`)
- Returns structured data (title, body, date)

**Why Guardian over other sources?**
- Free API with generous limits
- High-quality sports journalism
- Well-documented and reliable
- Used in academic research

**Rate Limiting**
- Free tier: 1 request per second
- We respect this with `time.sleep(1)`

In [ ]:
# ============================================================
# GUARDIAN API SCRAPER
# ============================================================

def fetch_guardian_page(from_date: str, to_date: str, page: int = 1) -> dict:
    """
    Fetch one page of Premier League articles from Guardian API.
    
    Args:
        from_date: Start date (YYYY-MM-DD)
        to_date: End date (YYYY-MM-DD)
        page: Page number (API returns 50 results per page)
    
    Returns:
        API response as dictionary
    """
    url = "https://content.guardianapis.com/search"
    
    params = {
        "api-key": GUARDIAN_API_KEY,
        "tag": "football/premierleague",  # IMPORTANT: Only PL-tagged articles
        "from-date": from_date,
        "to-date": to_date,
        "page": page,
        "page-size": 50,
        "show-fields": "headline,trailText,body,byline",
        "order-by": "newest",
    }
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def scrape_season(season: str, max_articles: int = 1000) -> list[dict]:
    """
    Scrape all Premier League articles for a season.
    
    Args:
        season: Season string (e.g., "2023-24")
        max_articles: Maximum articles to fetch (for testing, use lower number)
    
    Returns:
        List of article dictionaries
    """
    from_date, to_date = SEASONS[season]
    
    print(f"\n📰 Scraping {season} ({from_date} to {to_date})...")
    
    articles = []
    page = 1
    
    while len(articles) < max_articles:
        try:
            result = fetch_guardian_page(from_date, to_date, page)
            response = result.get("response", {})
            
            page_articles = response.get("results", [])
            total_pages = response.get("pages", 1)
            total_results = response.get("total", 0)
            
            if page == 1:
                print(f"   Total available: {total_results} articles")
            
            if not page_articles:
                break
            
            # Process each article
            for article in page_articles:
                fields = article.get("fields", {})
                articles.append({
                    "article_id": article.get("id", ""),
                    "title": fields.get("headline", article.get("webTitle", "")),
                    "summary": fields.get("trailText", ""),
                    "body_html": fields.get("body", ""),
                    "published": article.get("webPublicationDate", ""),
                    "url": article.get("webUrl", ""),
                    "season": season,
                })
            
            print(f"   Page {page}/{total_pages}: {len(articles)} articles collected")
            
            if page >= total_pages:
                break
                
            page += 1
            time.sleep(1)  # Rate limiting - be a good API citizen!
            
        except Exception as e:
            print(f"   Error on page {page}: {e}")
            break
    
    return articles[:max_articles]


print("✓ Scraper functions defined")

In [ ]:
# ============================================================
# SCRAPE ARTICLES FOR BOTH SEASONS
# ============================================================

# Set max_articles per season (increase for production, decrease for testing)
MAX_ARTICLES_PER_SEASON = 500  # Adjust based on your needs

all_articles = []

for season in SEASONS.keys():
    season_articles = scrape_season(season, max_articles=MAX_ARTICLES_PER_SEASON)
    all_articles.extend(season_articles)
    
    # Save intermediate result
    output_file = RAW_NEWS_DIR / f"guardian_{season}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(season_articles, f, indent=2, ensure_ascii=False)
    print(f"   Saved to {output_file}")

print(f"\n✓ Total articles scraped: {len(all_articles)}")

---

## 3️⃣ Clean & Parse Articles

**What is HTML parsing?**

The Guardian API returns article bodies as HTML:
```html
<p>Mohamed Salah scored a <strong>brilliant</strong> goal...</p>
```

We need plain text for NLP:
```
Mohamed Salah scored a brilliant goal...
```

**BeautifulSoup** is the standard library for this - it removes tags while preserving text.

**Why extract "first paragraph"?**

News articles follow the "inverted pyramid" structure:
- First paragraph: Most important facts (who, what, when)
- Later paragraphs: Background details

Player mentions in the first paragraph are more likely to be the article's main subject.

In [ ]:
# ============================================================
# TEXT CLEANING FUNCTIONS
# ============================================================

def html_to_text(html: str) -> str:
    """
    Convert HTML to plain text.
    
    Uses BeautifulSoup to:
    1. Remove script/style tags (JavaScript, CSS)
    2. Extract text content from remaining tags
    3. Normalize whitespace
    """
    if not html:
        return ""
    
    soup = BeautifulSoup(html, "lxml")
    
    # Remove non-content elements
    for element in soup(["script", "style", "aside", "nav", "footer"]):
        element.decompose()
    
    # Get text with spaces between elements
    text = soup.get_text(separator=" ")
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def extract_first_paragraph(text: str, num_sentences: int = 3) -> str:
    """
    Extract the first few sentences.
    
    These are most important for:
    - Determining article subject
    - Sentiment analysis (key claims)
    """
    if not text:
        return ""
    
    # Split on sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(sentences[:num_sentences])


# Keywords indicating non-Premier League content
# (Safety net for articles that slipped through tag filter)
NON_PL_KEYWORDS = [
    "women's", "wsl", "nwsl", "lionesses",
    "women's super league", "women's world cup",
]

def is_pl_relevant(title: str, body: str) -> bool:
    """Check if article is about men's Premier League."""
    text_lower = f"{title} {body}".lower()
    return not any(kw in text_lower for kw in NON_PL_KEYWORDS)


print("✓ Text cleaning functions defined")

# Test
test_html = "<p>Mohamed <strong>Salah</strong> scored twice.</p>"
print(f"\nTest: '{test_html}' → '{html_to_text(test_html)}'")

In [ ]:
# ============================================================
# PROCESS ALL ARTICLES
# ============================================================

processed_articles = []
filtered_count = 0

for article in tqdm(all_articles, desc="Processing articles"):
    # Convert HTML to text
    body_text = html_to_text(article.get("body_html", ""))
    title = article.get("title", "")
    
    # Skip if too short (likely not a real article)
    if len(body_text) < 100:
        filtered_count += 1
        continue
    
    # Skip non-PL content
    if not is_pl_relevant(title, body_text):
        filtered_count += 1
        continue
    
    processed_articles.append({
        "article_id": article["article_id"],
        "title": title,
        "body": body_text,
        "first_paragraph": extract_first_paragraph(body_text),
        "published": article["published"],
        "season": article["season"],
        "url": article["url"],
    })

print(f"\n✓ Processed {len(processed_articles)} articles (filtered {filtered_count})")

# Show sample
if processed_articles:
    sample = processed_articles[0]
    print(f"\nSample article:")
    print(f"  Title: {sample['title'][:80]}...")
    print(f"  First paragraph: {sample['first_paragraph'][:100]}...")

---

## 4️⃣ Build Player Knowledge Base

**The Entity Linking Problem**

News articles mention players by name:
- "Salah scored a hat-trick"
- "Mohamed Salah was brilliant"
- "Mo Salah leads the golden boot race"

FPL uses numeric IDs:
- Mohamed Salah = `element_id 308`

We need a **Knowledge Base** that maps:
```
"salah" → 308
"mohamed salah" → 308
"mo salah" → 308
```

**Challenges:**
1. **Ambiguity**: "Silva" could be Bernardo (Man City) or others
2. **Name variations**: Same player, different spellings
3. **Season changes**: FPL reuses element IDs across seasons!

In [ ]:
# ============================================================
# KNOWLEDGE BASE BUILDER
# ============================================================

def normalize_name(name: str) -> str:
    """
    Normalize a player name for matching.
    
    Examples:
        "Mohamed_Salah" → "mohamed salah"
        "HAALAND" → "haaland"
    """
    if not name or str(name) == "nan":
        return ""
    name = str(name)
    name = re.sub(r"_\d+$", "", name)  # Remove trailing IDs
    name = name.replace("_", " ")
    name = re.sub(r"\s+", " ", name.lower().strip())
    return name


def generate_name_variants(full_name: str) -> list[str]:
    """
    Generate name variants that might appear in articles.
    
    "mohamed salah" → ["mohamed salah", "salah", "m. salah"]
    """
    if not full_name:
        return []
    
    variants = [full_name]
    parts = full_name.split()
    
    if len(parts) >= 2:
        # Last name only (most common in articles)
        last_name = parts[-1]
        if len(last_name) >= MIN_NAME_LENGTH:
            variants.append(last_name)
        
        # First + Last (handles middle names)
        first_last = f"{parts[0]} {parts[-1]}"
        if first_last != full_name:
            variants.append(first_last)
        
        # Initial + Last: "M. Salah"
        initial_last = f"{parts[0][0]}. {parts[-1]}"
        variants.append(initial_last)
    
    return variants


def build_knowledge_base(fpl_path: Path, seasons: list[str]) -> dict:
    """
    Build a player name → element ID mapping.
    
    Returns:
        {
            "players": {"308_2024-25": {name, team, ...}},
            "name_to_elements": {"salah": [(308, "2024-25"), ...]}
        }
    """
    print("Loading FPL data...")
    df = pd.read_csv(fpl_path, low_memory=False)
    df = df[df["season"].isin(seasons)]
    
    # Get unique players
    players_df = df[["element", "name", "team", "position", "season"]].drop_duplicates()
    print(f"  Found {len(players_df)} unique player-seasons")
    
    players = {}
    name_to_elements = defaultdict(list)
    
    for _, row in tqdm(players_df.iterrows(), total=len(players_df), desc="Building KB"):
        element = int(row["element"])
        season = row["season"]
        player_key = f"{element}_{season}"
        
        full_name = normalize_name(row["name"])
        if not full_name:
            continue
        
        players[player_key] = {
            "element": element,
            "name": full_name,
            "team": str(row.get("team", "")),
            "position": str(row.get("position", "")),
            "season": season,
        }
        
        # Map all name variants to this element
        for variant in generate_name_variants(full_name):
            if (element, season) not in name_to_elements[variant]:
                name_to_elements[variant].append((element, season))
    
    # Remove highly ambiguous names (>3 players with same name in a season)
    filtered_names = {}
    for name, mappings in name_to_elements.items():
        season_counts = defaultdict(int)
        for _, season in mappings:
            season_counts[season] += 1
        if max(season_counts.values()) <= 3:
            filtered_names[name] = mappings
    
    print(f"\n✓ Knowledge Base built!")
    print(f"  Players: {len(players)}")
    print(f"  Name variants: {len(filtered_names)}")
    
    return {"players": players, "name_to_elements": filtered_names}


# Build the KB
kb = build_knowledge_base(FPL_DATA_PATH, list(SEASONS.keys()))

In [ ]:
# ============================================================
# TEST THE KNOWLEDGE BASE
# ============================================================

print("Testing Knowledge Base lookups:")

test_names = ["salah", "haaland", "palmer", "saka"]

for name in test_names:
    matches = kb["name_to_elements"].get(name, [])
    if matches:
        for elem, season in matches[:2]:  # Show first 2
            player = kb["players"].get(f"{elem}_{season}", {})
            print(f"  '{name}' → {player.get('name', 'N/A')} ({player.get('team', 'N/A')}, {season})")
    else:
        print(f"  '{name}' → No matches (may be ambiguous or short name)")

---

## 5️⃣ Entity Linking: Find Player Mentions

**Named Entity Recognition (NER)**

spaCy's NER model identifies:
- `PERSON`: Mohamed Salah, Pep Guardiola
- `ORG`: Liverpool, Manchester City
- `GPE`: England, Manchester

We use NER + Knowledge Base matching:
1. spaCy finds "Salah" and labels it as PERSON
2. We look up "salah" in our KB
3. We find element 308 (Mohamed Salah)

**Relevance Scoring**

Not all mentions are equal:
- Title mention = 3x weight (article is ABOUT this player)
- First paragraph = 2x weight (player is important)
- Body mention = 1x weight (player is mentioned)

In [ ]:
# ============================================================
# ENTITY LINKING
# ============================================================

# Load spaCy model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")
print(f"✓ Loaded: {nlp.meta['name']}")


def find_player_mentions(text: str, season: str) -> dict[int, float]:
    """
    Find player mentions in text using spaCy NER + KB lookup.
    
    Returns:
        {element_id: mention_count}
    """
    if not text:
        return {}
    
    doc = nlp(text)
    mentions = defaultdict(float)
    
    # Find PERSON entities via NER
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            name = normalize_name(ent.text)
            matches = kb["name_to_elements"].get(name, [])
            
            for elem, match_season in matches:
                # Prefer same-season matches
                weight = 1.0 if match_season == season else 0.5
                mentions[elem] += weight
    
    # Also do direct KB lookup (catches names spaCy might miss)
    text_lower = text.lower()
    for name, mappings in kb["name_to_elements"].items():
        if len(name) >= MIN_NAME_LENGTH:
            pattern = r"\b" + re.escape(name) + r"\b"
            count = len(re.findall(pattern, text_lower))
            if count > 0:
                for elem, match_season in mappings:
                    if match_season == season:
                        mentions[elem] += count * 0.5
    
    return dict(mentions)


# Test NER
test_text = "Mohamed Salah scored twice as Liverpool beat Manchester City. Haaland was quiet."
doc = nlp(test_text)

print(f"\nNER test: '{test_text}'")
print("Entities:")
for ent in doc.ents:
    print(f"  {ent.text} → {ent.label_}")

In [ ]:
# ============================================================
# LINK ALL ARTICLES TO PLAYERS
# ============================================================

# Relevance weights
TITLE_WEIGHT = 3.0
FIRST_PARA_WEIGHT = 2.0
BODY_WEIGHT = 1.0
MIN_RELEVANCE = 1.0  # Minimum score to keep

alignments = []

for article in tqdm(processed_articles, desc="Linking articles to players"):
    season = article["season"]
    
    # Find mentions in each section
    title_mentions = find_player_mentions(article["title"], season)
    first_para_mentions = find_player_mentions(article["first_paragraph"], season)
    body_mentions = find_player_mentions(article["body"], season)
    
    # Combine all unique players mentioned
    all_elements = set(title_mentions) | set(first_para_mentions) | set(body_mentions)
    
    for element in all_elements:
        # Calculate weighted relevance score
        score = (
            title_mentions.get(element, 0) * TITLE_WEIGHT +
            first_para_mentions.get(element, 0) * FIRST_PARA_WEIGHT +
            body_mentions.get(element, 0) * BODY_WEIGHT
        )
        
        if score >= MIN_RELEVANCE:
            alignments.append({
                "article_id": article["article_id"],
                "element": element,
                "season": season,
                "relevance_score": round(score, 2),
                "in_title": element in title_mentions,
                "published": article["published"],
            })

alignments_df = pd.DataFrame(alignments)

print(f"\n✓ Created {len(alignments_df)} player-article alignments")
print(f"  Unique articles: {alignments_df['article_id'].nunique()}")
print(f"  Unique players: {alignments_df['element'].nunique()}")

# Show top alignments
print("\nTop 10 alignments by relevance:")
top_alignments = alignments_df.nlargest(10, "relevance_score")
for _, row in top_alignments.iterrows():
    player = kb["players"].get(f"{row['element']}_{row['season']}", {})
    print(f"  {player.get('name', 'N/A')}: score={row['relevance_score']}, in_title={row['in_title']}")

---

## 6️⃣ Sentiment Analysis

**Why Sentiment Matters for FPL**

Research shows:
- Negative news (injuries, suspensions, manager conflicts) → Lower points
- Positive news (form, confidence, team chemistry) → Higher points

The FPL Transformer paper achieved 10% MSE improvement using sentiment.

**Model: cardiffnlp/twitter-roberta-base-sentiment**

- Pre-trained on 58M tweets + news
- Outputs: positive, negative, neutral probabilities
- Fast inference with GPU

In [ ]:
# ============================================================
# SENTIMENT ANALYSIS
# ============================================================

print("Loading sentiment model (this may take a moment)...")

# Use Cardiff NLP's RoBERTa model - well-tested and reliable
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    max_length=512,
    truncation=True,
)

print("✓ Sentiment model loaded!")

# Test
test_texts = [
    "Salah scores brilliant hat-trick in dominant Liverpool win!",
    "Injury concerns for Haaland ahead of crucial match",
    "Liverpool face Manchester City on Saturday",
]

print("\nSentiment test:")
for text in test_texts:
    result = sentiment_model(text)[0]
    print(f"  '{text[:50]}...' → {result['label']} ({result['score']:.2f})")

In [ ]:
# ============================================================
# ANALYZE SENTIMENT FOR ALL ARTICLES
# ============================================================

article_sentiments = []

for article in tqdm(processed_articles, desc="Analyzing sentiment"):
    # Use title + first paragraph (most relevant for sentiment)
    text = f"{article['title']}. {article['first_paragraph']}"[:512]
    
    try:
        result = sentiment_model(text)[0]
        label = result["label"].lower()
        score = result["score"]
        
        # Convert to probability distribution
        if "positive" in label:
            pos, neg, neu = score, 0.0, 1 - score
        elif "negative" in label:
            pos, neg, neu = 0.0, score, 1 - score
        else:  # neutral
            pos, neg, neu = 0.0, 0.0, score
            
    except Exception as e:
        # Default to neutral on error
        pos, neg, neu = 0.33, 0.33, 0.34
    
    article_sentiments.append({
        "article_id": article["article_id"],
        "sentiment_positive": pos,
        "sentiment_negative": neg,
        "sentiment_neutral": neu,
    })

sentiment_df = pd.DataFrame(article_sentiments)

print(f"\n✓ Analyzed {len(sentiment_df)} articles")
print(f"\nSentiment distribution:")
print(f"  Avg positive: {sentiment_df['sentiment_positive'].mean():.3f}")
print(f"  Avg negative: {sentiment_df['sentiment_negative'].mean():.3f}")
print(f"  Avg neutral: {sentiment_df['sentiment_neutral'].mean():.3f}")

---

## 7️⃣ Generate Text Embeddings

**What are Embeddings?**

Embeddings convert text into dense numerical vectors:
```
"Salah scores brilliant goal" → [0.12, -0.34, 0.56, ...] (384 numbers)
```

**Why they're useful:**
- Capture semantic meaning (not just keywords)
- Similar articles → similar vectors
- Can be directly used as ML features

**Model: all-MiniLM-L6-v2**
- 384 dimensions (compact but effective)
- Fast inference
- Good balance of speed/quality
- Industry standard for semantic search

In [ ]:
# ============================================================
# EMBEDDING GENERATION
# ============================================================

print(f"Loading embedding model: {EMBEDDING_MODEL}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✓ Loaded! Dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Prepare texts (title + first paragraph captures the essence)
article_texts = [
    f"{a['title']}. {a['first_paragraph']}"
    for a in processed_articles
]
article_ids = [a["article_id"] for a in processed_articles]

# Generate embeddings
print(f"\nGenerating embeddings for {len(article_texts)} articles...")
embeddings = embedding_model.encode(
    article_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"\n✓ Generated embeddings: {embeddings.shape}")

# Create article_id to embedding index mapping
id_to_embedding_idx = {aid: i for i, aid in enumerate(article_ids)}

---

## 8️⃣ Aggregate to Player-Level Features

**The Final Step**

We now have:
- Which articles mention which players (alignments)
- Sentiment for each article
- Embeddings for each article

We need to aggregate to **one row per player per season**:

```
Player: Salah, Season: 2024-25
├── Article 1: sentiment_pos=0.8, embedding=[...]
├── Article 2: sentiment_pos=0.6, embedding=[...]
└── Article 3: sentiment_pos=0.9, embedding=[...]

Aggregated:
├── news_article_count = 3
├── news_sentiment_positive = mean(0.8, 0.6, 0.9) = 0.77
└── news_embedding = mean([...], [...], [...]) = [...]
```

In [ ]:
# ============================================================
# AGGREGATE FEATURES
# ============================================================

# Merge alignments with sentiment
merged_df = alignments_df.merge(sentiment_df, on="article_id", how="left")

# Fill missing sentiment with neutral
merged_df["sentiment_positive"] = merged_df["sentiment_positive"].fillna(0.33)
merged_df["sentiment_negative"] = merged_df["sentiment_negative"].fillna(0.33)
merged_df["sentiment_neutral"] = merged_df["sentiment_neutral"].fillna(0.34)

# Group by player-season
grouped = merged_df.groupby(["element", "season"])

player_features = []

for (element, season), group in tqdm(grouped, desc="Aggregating features"):
    # Count features
    article_count = len(group)
    title_mentions = group["in_title"].sum()
    avg_relevance = group["relevance_score"].mean()
    
    # Sentiment features (mean across articles)
    avg_pos = group["sentiment_positive"].mean()
    avg_neg = group["sentiment_negative"].mean()
    avg_neu = group["sentiment_neutral"].mean()
    
    # Embedding features (mean pooling)
    article_indices = [
        id_to_embedding_idx[aid]
        for aid in group["article_id"]
        if aid in id_to_embedding_idx
    ]
    
    if article_indices:
        player_embedding = embeddings[article_indices].mean(axis=0)
    else:
        player_embedding = np.zeros(EMBEDDING_DIM)
    
    # Build feature row
    features = {
        "element": element,
        "season": season,
        "news_article_count": article_count,
        "news_title_mentions": title_mentions,
        "news_avg_relevance": round(avg_relevance, 3),
        "news_sentiment_positive": round(avg_pos, 4),
        "news_sentiment_negative": round(avg_neg, 4),
        "news_sentiment_neutral": round(avg_neu, 4),
    }
    
    # Add embedding dimensions
    for i, val in enumerate(player_embedding):
        features[f"news_emb_{i}"] = round(float(val), 6)
    
    player_features.append(features)

features_df = pd.DataFrame(player_features)

print(f"\n✓ Created features for {len(features_df)} player-season combinations")

In [ ]:
# ============================================================
# SAVE FINAL OUTPUT
# ============================================================

# Save to CSV
features_df.to_csv(OUTPUT_PATH, index=False)
print(f"✓ Saved to {OUTPUT_PATH}")

# Summary
print(f"\n" + "="*60)
print("PIPELINE COMPLETE!")
print("="*60)

print(f"\nOutput: {OUTPUT_PATH}")
print(f"  Rows: {len(features_df)}")
print(f"  Columns: {len(features_df.columns)}")

print(f"\nFeature breakdown:")
print(f"  - Count features: 3 (article_count, title_mentions, avg_relevance)")
print(f"  - Sentiment features: 3 (positive, negative, neutral)")
print(f"  - Embedding features: {EMBEDDING_DIM}")
print(f"  - Total: {3 + 3 + EMBEDDING_DIM} features per player")

print(f"\nSample (first 5 players):")
display_cols = ["element", "season", "news_article_count", 
                "news_sentiment_positive", "news_sentiment_negative"]
print(features_df[display_cols].head())

In [ ]:
# ============================================================
# DOWNLOAD OUTPUT (Colab only)
# ============================================================

if IN_COLAB:
    print("📥 Downloading news_features.csv...")
    from google.colab import files
    files.download(str(OUTPUT_PATH))
    print("\n✓ Download started! Save this file to your project's data/features/ folder.")
else:
    print(f"\n✓ File saved locally at: {OUTPUT_PATH}")
    print("\nNext step: Merge with your FPL data for model training.")

---

## ✅ Summary

**What this pipeline does:**

1. **Scrapes** Premier League articles from Guardian API
2. **Cleans** HTML to plain text
3. **Links** articles to FPL players using NER + Knowledge Base
4. **Extracts sentiment** (positive/negative/neutral)
5. **Generates embeddings** (384-dim semantic vectors)
6. **Aggregates** to player-level features

**Output:** `news_features.csv` ready to merge with FPL structured data

**Next steps:**
1. Merge news features with `fpl_base.csv` on `(element, season)`
2. Train model with and without news features
3. Compare performance (ablation study)

---

**Research citations:**
- ESPN/IBM Watson: https://arxiv.org/pdf/2111.02874
- FPL Transformer: https://reference-global.com/article/10.2478/ijcss-2025-0008